<a href="https://colab.research.google.com/github/tommihab/Haberstock-et-al.-2026-Supplementary-Material/blob/main/Google%20Colab%20Jupyter%20Notebooks/02_Haberstock_Ems_RF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Second part of the notebook series: RF model development

In [ ]:
# import packages

import os
import random
import numpy as np
import pandas as pd

In [ ]:
# set seed

SEED = 100

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
from pathlib import Path

# Define the GitHub repository
REPO_URL = "https://github.com/tommihab/Haberstock-et-al.-2026-Supplementary-Material.git"
REPO_DIR = Path("/content/Ems-Discharge-Forecasting")

# Clone the repository into Google Colab
if not REPO_DIR.exists():
    !git clone "{REPO_URL}" "{REPO_DIR}"

# Define the path to the CSV file inside the repository
file_path = REPO_DIR / "ML_Input_Data" / "Einen_Ems_Precipitation_Discharge_DOY.csv"

# Load the CSV file into a pandas DataFrame
data = pd.read_csv(file_path, sep=";")

# Convert the MESS_DATUM column to datetime objects
data["MESS_DATUM"] = pd.to_datetime(data["MESS_DATUM"])

# Sanity check
print("File loaded from:", file_path)
print("Data shape:", data.shape)
print(data.head())

Cloning into '/content/Ems-Discharge-Forecasting'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 48 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 15.65 MiB | 7.70 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Updating files: 100% (26/26), done.
File loaded from: /content/Ems-Discharge-Forecasting/ML_Input_Data/Einen_Ems_Precipitation_Discharge_DOY.csv
Data shape: (149040, 11)
           MESS_DATUM  augustdorf_2007-2024.csv  \
0 2007-01-01 00:00:00                      0.24   
1 2007-01-01 01:00:00                      0.03   
2 2007-01-01 02:00:00                      0.00   
3 2007-01-01 03:00:00                      0.03   
4 2007-01-01 04:00:00                      1.90   

   bielefeld-mitte_2007-2024.csv  delbrueck-steinhorst_2007-2024.csv  \
0                            0.2                                0.25   
1                  

In [ ]:
# Define columns

# Target variable
target_col = "discharge"

# Cyclical features (remain unchanged)
seasonal_cols = ["DOY_sin", "DOY_cos"]

# Define prediction horizons
HORIZONS = [3, 6, 12, 24, 72]
future_target_cols = [f"{target_col}_h{h}" for h in HORIZONS]

# All precipitation stations (everything except discharge, DOY and MESS_DATUM)
# MESS_DATUM should not be lagged, as it will become the index later and its cyclical components have already been extracted.
rain_cols = [c for c in data.columns if c not in [target_col] + seasonal_cols + ['MESS_DATUM']]

print("Precipitation columns:")
print(rain_cols)
print("\nPrediction horizons (in hours):")
print(HORIZONS)
print("\nNames of future target columns:")
print(future_target_cols)

Precipitation columns:
['augustdorf_2007-2024.csv', 'bielefeld-mitte_2007-2024.csv', 'delbrueck-steinhorst_2007-2024.csv', 'rheda-wiedenbrueck_2007-2024.csv', 'rietberg_2007-2024.csv', 'Versmold-Vorbruch_2007-2025.csv', 'warendorf-freckenhorst_2007-2024.csv']

Prediction horizons (in hours):
[3, 6, 12, 24, 72]

Names of future target columns:
['discharge_h3', 'discharge_h6', 'discharge_h12', 'discharge_h24', 'discharge_h72']


In [ ]:
# Lag-Feature Generation with concat

def add_lags_fast(df, cols, max_lag):
    """
    Generates lag features efficiently for given columns.
    """
    lag_blocks = []
    for lag in range(1, max_lag + 1):
        shifted = df[cols].shift(lag)
        shifted = shifted.add_suffix(f"_lag{lag}")
        lag_blocks.append(shifted)

    lag_df = pd.concat(lag_blocks, axis=1)
    out = pd.concat([df, lag_df], axis=1)

    # defragment
    out = out.copy()
    return out

In [ ]:
# Generate Lags
LOOKBACK = 96

data_lagged = data.copy()

# Precipitation Lags
data_lagged = add_lags_fast(data_lagged, cols=rain_cols, max_lag=LOOKBACK)

# Discharge Lags (as input)
data_lagged = add_lags_fast(data_lagged, cols=[target_col], max_lag=LOOKBACK)

print("Shape with lags (incl. NaNs):", data_lagged.shape)

Shape with lags (incl. NaNs): (149040, 779)


In [ ]:
# remove NaNs

data_lagged_clean = data_lagged.dropna()

print("Shape nach Entfernen der NaNs:", data_lagged_clean.shape)


Shape nach Entfernen der NaNs: (148944, 779)


In [ ]:
# Overview over final feature table
display(pd.DataFrame({
    "Spalte": data_lagged_clean.columns,
    "Typ": data_lagged_clean.dtypes.values
}))


,Spalte,Typ
0,MESS_DATUM,datetime64[ns]
1,augustdorf_2007-2024.csv,float64
2,bielefeld-mitte_2007-2024.csv,float64
3,delbrueck-steinhorst_2007-2024.csv,float64
4,rheda-wiedenbrueck_2007-2024.csv,float64
...,...,...
774,discharge_lag92,float64
775,discharge_lag93,float64
776,discharge_lag94,float64
777,discharge_lag95,float64


In [ ]:
# View a few consecutive timestamps
display(data_lagged_clean.iloc[:5])

,MESS_DATUM,augustdorf_2007-2024.csv,bielefeld-mitte_2007-2024.csv,delbrueck-steinhorst_2007-2024.csv,rheda-wiedenbrueck_2007-2024.csv,rietberg_2007-2024.csv,Versmold-Vorbruch_2007-2025.csv,warendorf-freckenhorst_2007-2024.csv,discharge,DOY_sin,...,discharge_lag87,discharge_lag88,discharge_lag89,discharge_lag90,discharge_lag91,discharge_lag92,discharge_lag93,discharge_lag94,discharge_lag95,discharge_lag96
96,2007-01-05 00:00:00,0.00,1.4,0.00,0.0,0.00,0.00,0.00,21.01,0.085906,...,14.68,14.65,14.63,14.60,14.57,14.55,14.52,14.50,14.47,14.45
97,2007-01-05 01:00:00,0.12,0.7,0.03,0.0,0.00,0.46,0.00,21.32,0.085906,...,14.70,14.68,14.65,14.63,14.60,14.57,14.55,14.52,14.50,14.47
98,2007-01-05 02:00:00,0.31,0.2,0.04,0.0,0.00,0.13,0.06,21.40,0.085906,...,14.73,14.70,14.68,14.65,14.63,14.60,14.57,14.55,14.52,14.50
99,2007-01-05 03:00:00,1.52,0.1,0.00,0.0,0.00,0.03,0.00,21.48,0.085906,...,14.88,14.73,14.70,14.68,14.65,14.63,14.60,14.57,14.55,14.52
100,2007-01-05 04:00:00,0.11,0.5,0.03,0.0,0.03,0.09,0.00,21.79,0.085906,...,15.03,14.88,14.73,14.70,14.68,14.65,14.63,14.60,14.57,14.55


In [ ]:
# Feature Split (X)

# Before splitting, create the future target variables by shifting the original target_col
for horizon in HORIZONS:
    future_col_name = f"{target_col}_h{horizon}"
    # Shift the target column ('discharge') by the specified horizon to create future targets
    data_lagged_clean[future_col_name] = data_lagged_clean[target_col].shift(-horizon)

# X contains all columns except the original target variable and the future target variables
X = data_lagged_clean.drop(columns=[target_col] + future_target_cols)

print("X shape:", X.shape)
# y shapes will be determined per horizon in the training loop

/tmp/ipykernel_2287/4014816974.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_lagged_clean[future_col_name] = data_lagged_clean[target_col].shift(-horizon)
/tmp/ipykernel_2287/4014816974.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_lagged_clean[future_col_name] = data_lagged_clean[target_col].shift(-horizon)
/tmp/ipykernel_2287/4014816974.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

X shape: (148944, 778)


In [ ]:
# Secure chronological sorting and set MESS_DATUM as index

# Check if 'MESS_DATUM' is still a column before setting it as index.
if 'MESS_DATUM' in data_lagged_clean.columns:
    data_lagged_clean = data_lagged_clean.set_index('MESS_DATUM')

# Ensure the index is sorted chronologically
data_lagged_clean = data_lagged_clean.sort_index()

print("Monotonically increasing:", data_lagged_clean.index.is_monotonic_increasing)

Monotonically increasing: True


In [ ]:
# Calculate split boundaries

n = len(data_lagged_clean)

idx_train_end = int(n * 0.60)
idx_val_end   = int(n * 0.75)
print("Total:", n)
print("Train until Index:", idx_train_end)
print("Val until Index:", idx_val_end)

Total: 148944
Train until Index: 89366
Val until Index: 111708


In [ ]:
# Chronological Split

train_data = data_lagged_clean.iloc[:idx_train_end]
val_data   = data_lagged_clean.iloc[idx_train_end:idx_val_end]
test_data  = data_lagged_clean.iloc[idx_val_end:]

In [ ]:
# Create X (y will be generated in the training loop for each horizon)

X_train = train_data.drop(columns=[target_col] + future_target_cols)
X_val = val_data.drop(columns=[target_col] + future_target_cols)
X_test = test_data.drop(columns=[target_col] + future_target_cols)

# Define y_train, y_val, y_test for the first horizon for display purposes

first_horizon_target_col = f"{target_col}_h{HORIZONS[0]}"
y_train = train_data[first_horizon_target_col]
y_val = val_data[first_horizon_target_col]
y_test = test_data[first_horizon_target_col]

# Display shapes to confirm
# Shapes
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

# Time periods
print("\nTime periods:")
print("Train:", X_train.index.min(), "->", X_train.index.max())
print("Val:  ", X_val.index.min(), "->", X_val.index.max())
print("Test: ", X_test.index.min(), "->", X_test.index.max())

Train: (89366, 777) (89366,)
Val:   (22342, 777) (22342,)
Test:  (37236, 777) (37236,)

Time periods:
Train: 2007-01-05 00:00:00 -> 2017-03-16 13:00:00
Val:   2017-03-16 14:00:00 -> 2019-10-03 11:00:00
Test:  2019-10-03 12:00:00 -> 2024-01-01 23:00:00


In [ ]:
# INFO: This is the code cell that runs the RF model. Running this code cell may take several hours. In order to save time you can scroll down to the CHECKPOINT.
# There, the model's results will be imported and the performance metrics caculated in a table
# Important: If this code cell is left out, the following code cell will fail because they are build upon eachother.

from sklearn.ensemble import RandomForestRegressor

param_candidates = [
    # "gentler" trees, more stability
    dict(n_estimators=500, max_features="sqrt", min_samples_leaf=2,  max_depth=30),
    dict(n_estimators=500, max_features=0.5,    min_samples_leaf=2,  max_depth=30),
    dict(n_estimators=500, max_features=0.3,    min_samples_leaf=2,  max_depth=30),

    # stronger regularisation
    dict(n_estimators=500, max_features="sqrt", min_samples_leaf=5,  max_depth=30),
    dict(n_estimators=500, max_features=0.3,    min_samples_leaf=5,  max_depth=30),

    # limit depth
    dict(n_estimators=500, max_features="sqrt", min_samples_leaf=2,  max_depth=20),
    dict(n_estimators=500, max_features=0.5,    min_samples_leaf=2,  max_depth=20),
    dict(n_estimators=500, max_features=0.3,    min_samples_leaf=2,  max_depth=20),

    # Optional: fewer samples per tree
    dict(n_estimators=500, max_features="sqrt", min_samples_leaf=2,  max_depth=30, bootstrap=True, max_samples=0.8),
    dict(n_estimators=500, max_features="sqrt", min_samples_leaf=5,  max_depth=30, bootstrap=True, max_samples=0.8),
]

# Dictionary to store trained models for each horizon
trained_models_by_horizon = {}

for horizon in HORIZONS:
    print(f"\n--- Training models for Horizon: H{horizon} ---")
    current_target_col = f"{target_col}_h{horizon}"

    # Get the target variables for the current horizon
    y_train_h = train_data[current_target_col]

    # List to store trained models for the current horizon
    trained_models_for_horizon = []

    for i, params in enumerate(param_candidates):
        print(f"\nTraining model {i+1}/{len(param_candidates)} for H{horizon} with parameters: {params}")
        model = RandomForestRegressor(**params, random_state=SEED, n_jobs=-1, verbose=1)
        model.fit(X_train, y_train_h)
        trained_models_for_horizon.append(model)
        print(f"Model {i+1} for H{horizon} training complete.")

    trained_models_by_horizon[horizon] = trained_models_for_horizon

print("\nAll models trained successfully for all horizons.")


--- Training models for Horizon: H3 ---

Training model 1/10 for H3 with parameters: {'n_estimators': 500, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'max_depth': 30}


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    4.6s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   23.2s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:   55.4s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:  1.1min finished


Model 1 for H3 training complete.

Training model 2/10 for H3 with parameters: {'n_estimators': 500, 'max_features': 0.5, 'min_samples_leaf': 2, 'max_depth': 30}


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:  5.6min
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed: 13.7min
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed: 15.9min finished


Model 2 for H3 training complete.

Training model 3/10 for H3 with parameters: {'n_estimators': 500, 'max_features': 0.3, 'min_samples_leaf': 2, 'max_depth': 30}


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   32.4s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:  2.8min
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:  6.7min
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:  7.9min finished


Model 5 for H6 training complete.

Training model 6/10 for H6 with parameters: {'n_estimators': 500, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'max_depth': 20}


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   18.6s
[Parallel(n_jobs=-1)]: Done 426 tasks      | elapsed:   44.0s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:   51.3s finished


Model 6 for H6 training complete.

Training model 7/10 for H6 with parameters: {'n_estimators': 500, 'max_features': 0.5, 'min_samples_leaf': 2, 'max_depth': 20}


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:   49.9s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:  4.2min


In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def nse_score(y_true, y_pred):
    # Drop NaN values for accurate calculation
    # Align y_true and y_pred by index, then drop NaNs
    combined = pd.DataFrame({'true': y_true, 'pred': y_pred}).dropna()
    y_true_clean = combined['true']
    y_pred_clean = combined['pred']

    numerator = np.sum((y_true_clean - y_pred_clean)**2)
    denominator = np.sum((y_true_clean - np.mean(y_true_clean))**2)

    if denominator == 0: # Avoid division by zero if y_true_clean is constant
        return 1.0 if numerator == 0 else -np.inf
    return 1 - (numerator / denominator)

def kge_score(y_true, y_pred):
    # Drop NaN values for accurate calculation
    # Align y_true and y_pred by index, then drop NaNs
    combined = pd.DataFrame({'true': y_true, 'pred': y_pred}).dropna()
    y_true_clean = combined['true']
    y_pred_clean = combined['pred']

    # Handle cases where after dropping NaNs, there are not enough samples for correlation
    if len(y_true_clean) < 2:
        return -np.inf # Or handle as appropriate for very few samples

    # Calculate correlation coefficient (r)
    r = np.corrcoef(y_true_clean, y_pred_clean)[0, 1]

    # Calculate mean ratio (beta)
    beta = np.mean(y_pred_clean) / np.mean(y_true_clean) if np.mean(y_true_clean) != 0 else np.inf

    # Calculate variability ratio (gamma) or alpha in some formulations
    gamma = np.std(y_pred_clean) / np.std(y_true_clean) if np.std(y_true_clean) != 0 else np.inf

    # KGE formula (version 2012)
    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)
    return kge

results = []

for horizon, trained_models_for_horizon in trained_models_by_horizon.items():
    print(f"\n--- Evaluating models for Horizon: H{horizon} ---")
    current_target_col = f"{target_col}_h{horizon}"

    # Get the target variables for the current horizon
    y_val_h = val_data[current_target_col]
    y_test_h = test_data[current_target_col]

    for i, model in enumerate(trained_models_for_horizon):
        model_name = f"Model {i+1} (H{horizon})"

        # Predict on validation set
        y_val_pred_array = model.predict(X_val)
        y_val_pred_series = pd.Series(y_val_pred_array, index=y_val_h.index)

        # Combine true and predicted values for validation and drop NaNs
        combined_val = pd.DataFrame({'true': y_val_h, 'pred': y_val_pred_series}).dropna()
        y_val_h_clean = combined_val['true']
        y_val_pred_clean = combined_val['pred']

        # Predict on test set
        y_test_pred_array = model.predict(X_test)
        y_test_pred_series = pd.Series(y_test_pred_array, index=y_test_h.index)

        # Combine true and predicted values for test and drop NaNs
        combined_test = pd.DataFrame({'true': y_test_h, 'pred': y_test_pred_series}).dropna()
        y_test_h_clean = combined_test['true']
        y_test_pred_clean = combined_test['pred']

        # Ensure there are samples left after dropping NaNs
        if len(y_val_h_clean) == 0 or len(y_test_h_clean) == 0:
            print(f"Skipping evaluation for {model_name} due to no valid samples after NaN removal.")
            continue

        # Calculate metrics for validation set using cleaned data
        r2_val = r2_score(y_val_h_clean, y_val_pred_clean)
        mae_val = mean_absolute_error(y_val_h_clean, y_val_pred_clean)
        rmse_val = np.sqrt(mean_squared_error(y_val_h_clean, y_val_pred_clean))
        nse_val = nse_score(y_val_h_clean, y_val_pred_clean) # Custom NSE uses cleaned data internally
        kge_val = kge_score(y_val_h_clean, y_val_pred_clean) # Custom KGE uses cleaned data internally

        # Calculate metrics for test set using cleaned data
        r2_test = r2_score(y_test_h_clean, y_test_pred_clean)
        mae_test = mean_absolute_error(y_test_h_clean, y_test_pred_clean)
        rmse_test = np.sqrt(mean_squared_error(y_test_h_clean, y_test_pred_clean))
        nse_test = nse_score(y_test_h_clean, y_test_pred_clean) # Custom NSE uses cleaned data internally
        kge_test = kge_score(y_test_h_clean, y_test_pred_clean) # Custom KGE uses cleaned data internally

        results.append({
            'Model': model_name,
            'Horizon': horizon,
            'Param_Index': i, # Add Param_Index here
            'R2_Val': r2_val,
            'MAE_Val': mae_val,
            'RMSE_Val': rmse_val,
            'NSE_Val': nse_val,
            'KGE_Val': kge_val,
            'R2_Test': r2_test,
            'MAE_Test': mae_test,
            'RMSE_Test': rmse_test,
            'NSE_Test': nse_test,
            'KGE_Test': kge_test
        })

results_df = pd.DataFrame(results)
print("\nModel Evaluation Results:")
display(results_df)

In [ ]:
# Find the best model per horizon based on NSE_Test
best_models_per_horizon = results_df.loc[results_df.groupby('Horizon')['NSE_Test'].idxmax()]

print("\nBestes Modell pro Horizont (basierend auf NSE_Test):")
display(best_models_per_horizon)

# Find the overall best model across all horizons based on NSE_Test
overall_best_model = results_df.loc[results_df['NSE_Test'].idxmax()]

print("\nInsgesamt bestes Modell (basierend auf NSE_Test):")
display(overall_best_model)

In [ ]:
# Identify the best compromise parameter set based on the average NSE_Test score
best_compromise_param_set = avg_performance_per_param_set.loc[avg_performance_per_param_set['Average_NSE_Test'].idxmax()]
print("Best compromise parameter set (based on average NSE_Test across all horizons):")
display(best_compromise_param_set)

In [ ]:
best_param_index = best_compromise_param_set['Param_Index']
best_compromise_performance = results_df[results_df['Param_Index'] == best_param_index]

print(f"Performance of the best compromise parameter set (Param_Index: {int(best_param_index)}) across all horizons:")
display(best_compromise_performance)

In [ ]:
# Retrieve and display the performance of the best compromise parameter set
best_param_index = best_compromise_param_set['Param_Index']
best_compromise_performance = results_df[results_df['Param_Index'] == best_param_index]

print(f"Performance of the best compromise parameter set (Param_Index: {int(best_param_index)}) across all horizons:")
display(best_compromise_performance)

In [ ]:
# CHECKPOINT: Run this code cell to load the RF model predictions and see the calculated performance metrics

import pandas as pd
import os
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# Redefine NSE and KGE functions if this cell might be run independently

def nse_score(y_true, y_pred):
    combined = pd.DataFrame({'true': y_true, 'pred': y_pred}).dropna()
    y_true_clean = combined['true']
    y_pred_clean = combined['pred']

    numerator = np.sum((y_true_clean - y_pred_clean)**2)
    denominator = np.sum((y_true_clean - np.mean(y_true_clean))**2)

    if denominator == 0: # Avoid division by zero if y_true_clean is constant
        return 1.0 if numerator == 0 else -np.inf
    return 1 - (numerator / denominator)

def kge_score(y_true, y_pred):
    combined = pd.DataFrame({'true': y_true, 'pred': y_pred}).dropna()
    y_true_clean = combined['true']
    y_pred_clean = combined['pred']

    if len(y_true_clean) < 2:
        return -np.inf

    r = np.corrcoef(y_true_clean, y_pred_clean)[0, 1]
    beta = np.mean(y_pred_clean) / np.mean(y_true_clean) if np.mean(y_true_clean) != 0 else np.inf
    gamma = np.std(y_pred_clean) / np.std(y_true_clean) if np.std(y_true_clean) != 0 else np.inf

    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)
    return kge


HORIZONS = [3, 6, 12, 24, 72]

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Define the directory containing the RF prediction files
rf_predictions_dir = REPO_DIR / "rf_predictions_best_compromise"

all_test_metrics = []

print("Loading and evaluating test set predictions for all horizons:")

for horizon in HORIZONS:
    test_file_path = (
        rf_predictions_dir
        / f"rf_predictions_H{horizon}_test.csv"
    )

    try:
        df_test_predictions = pd.read_csv(test_file_path)

        y_true = df_test_predictions["y_true"]
        y_pred = df_test_predictions["y_pred"]

        # Calculate metrics
        mae = mean_absolute_error(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        nse = nse_score(y_true, y_pred)
        kge = kge_score(y_true, y_pred)

        all_test_metrics.append({
            "Horizon": horizon,
            "MAE_Test": mae,
            "RMSE_Test": rmse,
            "NSE_Test": nse,
            "KGE_Test": kge
        })

        print(f"  - Metrics calculated for H{horizon}")

    except FileNotFoundError:
        print(
            f"  - Error: Test prediction file not found "
            f"for H{horizon} at {test_file_path}"
        )

    except Exception as e:
        print(
            f"  - An error occurred while processing "
            f"H{horizon} predictions: {e}"
        )

if all_test_metrics:
    df_all_test_metrics = pd.DataFrame(all_test_metrics)

    print("\n--- Overall Test Set Performance Metrics ---")
    display(df_all_test_metrics)

else:
    print(
        "No test set prediction files were successfully "
        "loaded and evaluated."
    )

Loading and evaluating test set predictions for all horizons:
  - Metrics calculated for H3
  - Metrics calculated for H6
  - Metrics calculated for H12
  - Metrics calculated for H24
  - Metrics calculated for H72

--- Overall Test Set Performance Metrics ---


,Horizon,MAE_Test,RMSE_Test,NSE_Test,KGE_Test
0,3,0.469042,2.075907,0.977684,0.929191
1,6,0.700139,2.727803,0.961470,0.904282
2,12,1.120016,3.689211,0.929532,0.867602
3,24,2.206176,5.697984,0.831942,0.791653
4,72,4.404820,9.100199,0.571813,0.638036
